In [1]:
from __future__ import annotations

import json
import sys
import warnings
from pathlib import Path
from typing import Any, Dict, List

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from IPython.display import Image, display

warnings.filterwarnings("ignore")

# Plot configuration
try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("default")

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "savefig.dpi": 300,
        "axes.titlesize": 13,
        "axes.titleweight": "bold",
        "axes.labelsize": 11,
        "axes.labelweight": "bold",
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "legend.fontsize": 10,
    }
)


class AgricultureDataPipeline:
    """End-to-end agricultural data science pipeline."""

    def __init__(
        self,
        data_path: str = "Cleaned_Agriculture_Data.json",
        output_dir: str = "output_analysis",
    ) -> None:
        self.data_path = Path(data_path)
        self.output_dir = Path(output_dir)
        self.figures_dir = self.output_dir / "figures"
        self.reports_dir = self.output_dir / "reports"
        self.df = pd.DataFrame()

        self.figures_dir.mkdir(parents=True, exist_ok=True)
        self.reports_dir.mkdir(parents=True, exist_ok=True)

    def _validate_columns(self, required: List[str]) -> None:
        """Raise a clear error when required columns are missing."""
        missing = [col for col in required if col not in self.df.columns]
        if missing:
            raise ValueError(
                "Dataset is missing required columns:\n"
                + "\n".join(f"  - {col}" for col in missing)
            )

    def _safe_divide(self, numerator: pd.Series, denominator: pd.Series) -> pd.Series:
        """Element-wise division that returns NaN for zero denominators."""
        return numerator.div(denominator.replace(0, np.nan))

    def load_and_validate_data(self) -> pd.DataFrame:
        """Load JSON, clean data, validate identifiers, and engineer features."""
        print("=" * 80)
        print("STEP 1: DATA INGESTION & QUALITY AUDIT")
        print("=" * 80)

        if not self.data_path.exists():
            candidates = [
                Path("Cleaned_Agriculture_Data.json"),
                Path("./Cleaned_Agriculture_Data.json"),
                Path("../Cleaned_Agriculture_Data.json"),
                Path("/content/Cleaned_Agriculture_Data.json"), # Added explicit Colab path
            ]
            found = next((path for path in candidates if path.exists()), None)
            if found is None:
                raise FileNotFoundError(
                    f"Cannot locate dataset at: {self.data_path.resolve()}\n"
                    "Please upload 'Cleaned_Agriculture_Data.json' to the Colab Files pane on the left."
                )
            self.data_path = found

        print(f"[INFO] Ingesting dataset from: {self.data_path}")

        with self.data_path.open("r", encoding="utf-8") as file:
            data = json.load(file)

        # Supports both a JSON list of records and {"data": [...]} format.
        if isinstance(data, dict):
            data = data.get("data", data)

        if not isinstance(data, list):
            raise ValueError(
                "Expected JSON data to be a list of records or a dictionary "
                "containing a 'data' list."
            )

        self.df = pd.DataFrame(data)

        if self.df.empty:
            raise ValueError("The dataset is empty.")

        print(
            f"[SUCCESS] Ingested {len(self.df):,} records "
            f"across {len(self.df.columns)} attributes."
        )

        required_columns = [
            "Farm_ID",
            "Crop",
            "Farm_Area_Hectares",
            "Production_Tonnes",
            "Market_Price_INR_Tonne",
            "Revenue_INR",
            "Total_Cost_INR",
            "Profit_INR",
            "Yield_Tonnes_Ha",
            "Water_Used_m3",
            "Water_Efficiency_t_per_1000m3",
            "Irrigation_Method",
            "Disease_Pest_Risk_pct",
            "Nitrogen_kg_ha",
            "Phosphorus_kg_ha",
            "Potassium_kg_ha",
        ]
        self._validate_columns(required_columns)

        # Convert expected numeric fields safely.
        numeric_candidates = [
            col
            for col in self.df.columns
            if col not in {"Farm_ID", "Crop", "Irrigation_Method", "State", "Season"}
        ]
        for col in numeric_candidates:
            self.df[col] = pd.to_numeric(self.df[col], errors="coerce")

        # Missing-value audit.
        null_counts_before = self.df.isna().sum()
        total_nulls = int(null_counts_before.sum())
        print(f"[AUDIT] Missing values before cleaning: {total_nulls}")

        # Data-science friendly cleaning:
        # median for numeric variables and mode for categorical variables.
        if total_nulls > 0:
            numeric_cols = self.df.select_dtypes(include=np.number).columns
            categorical_cols = self.df.select_dtypes(exclude=np.number).columns

            for col in numeric_cols:
                if self.df[col].isna().any():
                    median_value = self.df[col].median()
                    if pd.notna(median_value):
                        self.df[col] = self.df[col].fillna(median_value)
                    else: # If median is NaN (e.g., column is all NaN), fill with 0.
                        self.df[col] = self.df[col].fillna(0)

            for col in categorical_cols:
                if self.df[col].isna().any():
                    mode = self.df[col].mode(dropna=True)
                    if not mode.empty:
                        self.df[col] = self.df[col].fillna(mode.iloc[0])
                    else: # If mode is empty (e.g., column is all NaN), fill with 'Unknown'.
                        self.df[col] = self.df[col].fillna('Unknown')

            print("[INFO] Missing values handled using median/mode imputation.")

        # Remove rows still containing missing values when a column is entirely null.
        remaining_nulls = int(self.df.isna().sum().sum())
        if remaining_nulls:
            before = len(self.df)
            self.df = self.df.dropna()
            print(
                f"[INFO] Removed {before - len(self.df):,} rows "
                "with unresolved missing values."
            )

        # Duplicate Farm_ID audit.
        duplicate_count = int(self.df.duplicated(subset=["Farm_ID"]).sum())
        print(f"[AUDIT] Duplicate Farm IDs: {duplicate_count}")

        if duplicate_count > 0:
            self.df = self.df.drop_duplicates(subset=["Farm_ID"], keep="first")
            print("[INFO] Duplicate Farm_ID records removed.")

        # Prevent invalid calculations.
        invalid_area = self.df["Farm_Area_Hectares"] <= 0
        if invalid_area.any():
            print(f"[AUDIT] Invalid farm areas removed: {invalid_area.sum():,}")
            self.df = self.df.loc[~invalid_area].copy()

        # Feature engineering.
        self.df["Calculated_Revenue"] = (
            self.df["Production_Tonnes"] * self.df["Market_Price_INR_Tonne"]
        )

        self.df["Calculated_Profit"] = (
            self.df["Revenue_INR"] - self.df["Total_Cost_INR"]
        )

        self.df["Profit_Per_Hectare"] = self._safe_divide(
            self.df["Profit_INR"], self.df["Farm_Area_Hectares"]
        )
        self.df["Cost_Per_Hectare"] = self._safe_divide(
            self.df["Total_Cost_INR"], self.df["Farm_Area_Hectares"]
        )
        self.df["Revenue_Per_Hectare"] = self._safe_divide(
            self.df["Revenue_INR"], self.df["Farm_Area_Hectares"]
        )

        self.df["Profit_Margin_Pct"] = np.where(
            self.df["Revenue_INR"] > 0,
            (self.df["Profit_INR"] / self.df["Revenue_INR"]) * 100,
            np.nan,
        )

        self.df["Is_Profitable"] = self.df["Profit_INR"] > 0

        self.df["NPK_Total_kg_ha"] = (
            self.df["Nitrogen_kg_ha"]
            + self.df["Phosphorus_kg_ha"]
            + self.df["Potassium_kg_ha"]
        )

        self.df["Water_Used_Per_Ha_m3"] = self._safe_divide(
            self.df["Water_Used_m3"], self.df["Farm_Area_Hectares"]
        )

        # Save cleaned/engineered data for reproducibility.
        cleaned_file = self.reports_dir / "cleaned_engineered_agriculture_data.csv"
        self.df.to_csv(cleaned_file, index=False)
        print(f"[SAVED] Cleaned data exported to: {cleaned_file}")

        return self.df

    def compute_summary_statistics(self) -> Dict[str, Any]:
        """Generate descriptive statistics and business benchmarks."""
        print("\n" + "=" * 80)
        print("STEP 2: EXPLORATORY STATISTICAL SUMMARY")
        print("=" * 80)

        numeric_cols = self.df.select_dtypes(include=np.number).columns
        desc_df = self.df[numeric_cols].describe().T
        desc_df["skewness"] = self.df[numeric_cols].skew()
        desc_df["kurtosis"] = self.df[numeric_cols].kurtosis()
        desc_df["IQR"] = desc_df["75%"] - desc_df["25%"]

        summary_file = self.reports_dir / "descriptive_statistics.csv"
        desc_df.to_csv(summary_file)
        print(f"[SAVED] Numerical summary exported to: {summary_file}")

        total_farms = len(self.df)
        total_area = self.df["Farm_Area_Hectares"].sum()
        total_production = self.df["Production_Tonnes"].sum()
        total_revenue = self.df["Revenue_INR"].sum()
        total_cost = self.df["Total_Cost_INR"].sum()
        net_profit = self.df["Profit_INR"].sum()
        profitable_pct = self.df["Is_Profitable"].mean() * 100

        print("\n--- KEY AGRONOMIC & FINANCIAL BENCHMARKS ---")
        print(f"Total Farms Analyzed        : {total_farms:,}")
        print(f"Total Cultivated Land       : {total_area:,.2f} ha")
        print(f"Total Production            : {total_production:,.2f} Tonnes")
        print(f"Total Sector Revenue        : INR {total_revenue:,.2f}")
        print(f"Total Sector Cost           : INR {total_cost:,.2f}")
        print(f"Net Sector Profit            : INR {net_profit:,.2f}")
        print(f"Profitable Farm Share       : {profitable_pct:.2f}%")
        print(
            f"Mean Yield                  : "
            f"{self.df['Yield_Tonnes_Ha'].mean():.2f} t/ha "
            f"(Median: {self.df['Yield_Tonnes_Ha'].median():.2f} t/ha)"
        )
        print(
            f"Mean Water Efficiency       : "
            f"{self.df['Water_Efficiency_t_per_1000m3'].mean():.3f} "
            f"t/1,000m³"
        )

        return {
            "total_farms": total_farms,
            "profitable_pct": profitable_pct,
            "net_profit": net_profit,
            "total_production": total_production,
        }

    def generate_crop_analysis(self) -> pd.DataFrame:
        """Benchmark crop-level operational and financial performance."""
        print("\n" + "=" * 80)
        print("STEP 3: CROP PERFORMANCE BENCHMARKING")
        print("=" * 80)

        crop_stats = (
            self.df.groupby("Crop")
            .agg(
                Farms=("Farm_ID", "count"),
                Total_Area_Ha=("Farm_Area_Hectares", "sum"),
                Mean_Yield_t_ha=("Yield_Tonnes_Ha", "mean"),
                Median_Yield_t_ha=("Yield_Tonnes_Ha", "median"),
                Mean_Market_Price=("Market_Price_INR_Tonne", "mean"),
                Mean_Cost_Ha=("Cost_Per_Hectare", "mean"),
                Mean_Revenue_Ha=("Revenue_Per_Hectare", "mean"),
                Mean_Profit_Ha=("Profit_Per_Hectare", "mean"),
                Median_Profit_Ha=("Profit_Per_Hectare", "median"),
                Profitable_Farms_Pct=("Is_Profitable", lambda x: x.mean() * 100),
                Mean_Water_Eff=("Water_Efficiency_t_per_1000m3", "mean"),
                Mean_Disease_Risk=("Disease_Pest_Risk_pct", "mean"),
            )
            .reset_index()
            .sort_values("Mean_Profit_Ha", ascending=False)
        )

        crop_file = self.reports_dir / "crop_performance_summary.csv"
        crop_stats.to_csv(crop_file, index=False)

        print(crop_stats.to_string(index=False))
        print(f"[SAVED] Crop analysis exported to: {crop_file}")
        return crop_stats

    def generate_irrigation_analysis(self) -> pd.DataFrame:
        """Benchmark irrigation methods by productivity, water, and profit."""
        print("\n" + "=" * 80)
        print("STEP 4: IRRIGATION EFFICIENCY & ECONOMIC AUDIT")
        print("=" * 80)

        irrig_stats = (
            self.df.groupby("Irrigation_Method")
            .agg(
                Farms=("Farm_ID", "count"),
                Mean_Yield=("Yield_Tonnes_Ha", "mean"),
                Mean_Water_Used=("Water_Used_m3", "mean"),
                Mean_Water_Per_Ha=("Water_Used_Per_Ha_m3", "mean"),
                Mean_Water_Efficiency=(
                    "Water_Efficiency_t_per_1000m3",
                    "mean",
                ),
                Mean_Cost_Ha=("Cost_Per_Hectare", "mean"),
                Mean_Profit_Ha=("Profit_Per_Hectare", "mean"),
                Profitable_Pct=("Is_Profitable", lambda x: x.mean() * 100),
                Mean_Disease_Risk=("Disease_Pest_Risk_pct", "mean"),
            )
            .reset_index()
            .sort_values("Mean_Water_Efficiency", ascending=False)
        )

        irrig_file = self.reports_dir / "irrigation_efficiency_summary.csv"
        irrig_stats.to_csv(irrig_file, index=False)

        print(irrig_stats.to_string(index=False))
        print(f"[SAVED] Irrigation analysis exported to: {irrig_file}")
        return irrig_stats

    def generate_visualizations(self) -> None:
        """Generate analytical plots with corrected axis handling."""
        print("\n" + "=" * 80)
        print("STEP 5: VISUAL STORYTELLING & PLOT GENERATION")
        print("=" * 80)

        non_cane = self.df[self.df["Crop"] != "Sugarcane"].copy()
        if non_cane.empty:
            non_cane = self.df.copy()

        # Figure 1: Crop distribution and yield.
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))

        crop_order = self.df["Crop"].value_counts().index
        sns.countplot(
            data=self.df,
            x="Crop",
            order=crop_order,
            ax=axes[0],
            color="steelblue",
        )
        axes[0].set_title("Farm Representation by Crop Type")
        axes[0].set_xlabel("Crop Type")
        axes[0].set_ylabel("Farm Count")
        axes[0].tick_params(axis="x", rotation=30)

        for patch in axes[0].patches:
            height = patch.get_height()
            axes[0].annotate(
                f"{int(height)}",
                (patch.get_x() + patch.get_width() / 2, height / 2),
                ha="center",
                va="center",
                color="white",
                fontweight="bold",
            )

        sns.boxplot(
            data=non_cane,
            x="Crop",
            y="Yield_Tonnes_Ha",
            ax=axes[1],
            color="lightgray",
        )
        axes[1].set_title("Yield Distribution (Non-Sugarcane Crops, t/ha)")
        axes[1].set_xlabel("Crop Type")
        axes[1].set_ylabel("Yield (t/ha)")
        axes[1].tick_params(axis="x", rotation=30)

        fig.tight_layout()
        fig.savefig(self.figures_dir / "01_crop_distribution_and_yield.png")

        # Figure 2: Financial performance by crop.
        fig, ax = plt.subplots(figsize=(12, 6))
        fin_crop = (
            self.df.groupby("Crop")[
                ["Cost_Per_Hectare", "Revenue_Per_Hectare", "Profit_Per_Hectare"]
            ]
            .mean()
            .sort_values("Profit_Per_Hectare", ascending=False)
        )
        fin_crop.plot(kind="bar", ax=ax, width=0.8)
        ax.set_title("Average Cost, Revenue & Profit per Hectare by Crop")
        ax.set_xlabel("Crop")
        ax.set_ylabel("INR / Hectare")
        ax.axhline(0, color="black", linestyle="--", linewidth=1)
        ax.legend(["Cost/ha", "Revenue/ha", "Profit/ha"], loc="upper right")
        plt.xticks(rotation=30)
        fig.tight_layout()
        fig.savefig(self.figures_dir / "02_financial_performance_by_crop.png")

        # Figure 3: Irrigation comparison.
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))

        irrigation_order = (
            self.df.groupby("Irrigation_Method")[
                "Water_Efficiency_t_per_1000m3"
            ]
            .mean()
            .sort_values(ascending=False)
            .index
        )

        sns.barplot(
            data=self.df,
            x="Irrigation_Method",
            y="Water_Efficiency_t_per_1000m3",
            estimator="mean",
            errorbar=None,
            order=irrigation_order,
            ax=axes[0],
            color="steelblue",
        )
        axes[0].set_title(
            "Mean Water Efficiency by Irrigation Method (t / 1,000 m³)"
        )
        axes[0].set_xlabel("Irrigation Method")
        axes[0].set_ylabel("Water Efficiency")

        sns.barplot(
            data=self.df,
            x="Irrigation_Method",
            y="Profit_Per_Hectare",
            estimator="mean",
            errorbar=None,
            order=irrigation_order,
            ax=axes[1],
            color="seagreen",
        )
        axes[1].set_title("Mean Profit per Hectare by Irrigation Method")
        axes[1].set_xlabel("Irrigation Method")
        axes[1].set_ylabel("Profit / ha (INR)")
        axes[1].axhline(0, color="black", linestyle="--", linewidth=0.8)

        fig.tight_layout()
        fig.savefig(self.figures_dir / "03_irrigation_efficiency_comparison.png")

        # Figure 4: Correlation heatmap.
        fig, ax = plt.subplots(figsize=(13, 10))
        corr_cols = [
            "Farm_Area_Hectares",
            "Rainfall_mm",
            "Avg_Temperature_C",
            "Humidity_pct",
            "Sunlight_Hours_Day",
            "Soil_pH",
            "Soil_Moisture_pct",
            "Nitrogen_kg_ha",
            "Phosphorus_kg_ha",
            "Potassium_kg_ha",
            "Fertilizer_kg_ha",
            "Pesticide_Litre_ha",
            "Seed_Quality_Score",
            "Yield_Tonnes_Ha",
            "Water_Efficiency_t_per_1000m3",
            "Disease_Pest_Risk_pct",
            "Cost_Per_Hectare",
            "Revenue_Per_Hectare",
            "Profit_Per_Hectare",
        ]
        available_corr_cols = [c for c in corr_cols if c in self.df.columns]

        corr_matrix = self.df[available_corr_cols].corr()
        mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

        sns.heatmap(
            corr_matrix,
            mask=mask,
            cmap="vlag",
            vmin=-1,
            vmax=1,
            annot=True,
            fmt=".2f",
            ax=ax,
            cbar_kws={"label": "Pearson r"},
        )
        ax.set_title("Multivariate Feature Correlation Heatmap")
        plt.xticks(rotation=45, ha="right")
        fig.tight_layout()
        fig.savefig(self.figures_dir / "04_correlation_heatmap.png")

        # Figure 5: Agronomic regression plots.
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        regression_specs = [
            ("Soil_pH", "Soil pH vs Yield (Non-Sugarcane)", axes[0]),
            ("Soil_Moisture_pct", "Soil Moisture (%) vs Yield", axes[1]),
            ("Fertilizer_kg_ha", "Fertilizer (kg/ha) vs Yield", axes[2]),
        ]

        for feature, title, axis in regression_specs:
            sns.regplot(
                data=non_cane,
                x=feature,
                y="Yield_Tonnes_Ha",
                ax=axis,
                scatter_kws={"alpha": 0.3},
                line_kws={"color": "red"},
            )
            axis.set_title(title)
            axis.set_xlabel(feature.replace("_", " "))
            axis.set_ylabel("Yield (t/ha)")

        fig.tight_layout()
        fig.savefig(self.figures_dir / "05_soil_and_fertilizer_vs_yield.png")

        # Figure 6: Seasonal climate impact.
        if "Season" in self.df.columns:
            fig, axes = plt.subplots(1, 3, figsize=(18, 5))

            sns.boxplot(
                data=self.df,
                x="Season",
                y="Rainfall_mm",
                ax=axes[0],
                color="lightblue",
            )
            axes[0].set_title("Rainfall Dynamics across Seasons (mm)")

            sns.boxplot(
                data=self.df,
                x="Season",
                y="Avg_Temperature_C",
                ax=axes[1],
                color="navajowhite",
            )
            axes[1].set_title("Average Temperature across Seasons (°C)")

            sns.barplot(
                data=self.df,
                x="Season",
                y="Profit_Per_Hectare",
                ax=axes[2],
                errorbar=None,
                color="seagreen",
            )
            axes[2].set_title("Mean Profit per Hectare by Season (INR)")

            fig.tight_layout()
            fig.savefig(self.figures_dir / "06_seasonal_climate_impact.png")

        # Figure 7: Revenue vs cost frontier.
        fig, ax = plt.subplots(figsize=(10, 8))
        sns.scatterplot(
            data=self.df,
            x="Total_Cost_INR",
            y="Revenue_INR",
            hue="Crop",
            style="Is_Profitable",
            alpha=0.75,
            s=60,
            ax=ax,
        )

        max_val = max(
            self.df["Total_Cost_INR"].max(),
            self.df["Revenue_INR"].max(),
        )
        ax.plot(
            [0, max_val],
            [0, max_val],
            "k--",
            linewidth=2,
            label="Break-Even Threshold (Profit = 0)",
        )
        ax.set_title("Revenue vs Total Cost Frontier")
        ax.set_xlabel("Total Cost (INR)")
        ax.set_ylabel("Total Revenue (INR)")
        ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")

        fig.tight_layout()
        fig.savefig(self.figures_dir / "07_profit_vs_cost_scatter.png")

        # Figure 8: Disease risk.
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))

        sns.regplot(
            data=self.df,
            x="Humidity_pct",
            y="Disease_Pest_Risk_pct",
            ax=axes[0],
            scatter_kws={"alpha": 0.25},
            line_kws={"color": "darkred"},
        )
        axes[0].set_title("Humidity (%) vs Disease/Pest Risk (%)")

        sns.boxplot(
            data=self.df,
            x="Crop",
            y="Disease_Pest_Risk_pct",
            ax=axes[1],
            color="lightgray",
        )
        axes[1].set_title("Disease Risk Across Crop Types")
        axes[1].tick_params(axis="x", rotation=30)

        fig.tight_layout()
        fig.savefig(self.figures_dir / "08_disease_pest_risk_analysis.png")

        # Figure 9: Geographic performance.
        if "State" in self.df.columns:
            fig, axes = plt.subplots(1, 2, figsize=(16, 6))

            state_order = (
                self.df.groupby("State")["Profit_Per_Hectare"]
                .mean()
                .sort_values(ascending=False)
                .index
            )
            sns.barplot(
                data=self.df,
                x="State",
                y="Profit_Per_Hectare",
                order=state_order,
                ax=axes[0],
                color="steelblue",
            )
            axes[0].set_title("Profitability per Hectare by State (INR)")
            axes[0].tick_params(axis="x", rotation=45)

            state_water = (
                self.df.groupby("State")[
                    "Water_Efficiency_t_per_1000m3"
                ]
                .mean()
                .sort_values(ascending=False)
                .index
            )
            sns.barplot(
                data=self.df,
                x="State",
                y="Water_Efficiency_t_per_1000m3",
                order=state_water,
                ax=axes[1],
                color="seagreen",
            )
            axes[1].set_title("Water Efficiency by State (t / 1,000 m³)")
            axes[1].tick_params(axis="x", rotation=45)

            fig.tight_layout()
            fig.savefig(self.figures_dir / "09_state_district_performance.png")

        # Figure 10: Top vs bottom profit cohorts.
        fig, ax = plt.subplots(figsize=(12, 6))
        q_low = self.df["Profit_Per_Hectare"].quantile(0.10)
        q_high = self.df["Profit_Per_Hectare"].quantile(0.90)

        top_cohort = self.df[
            self.df["Profit_Per_Hectare"] >= q_high
        ].copy()
        bottom_cohort = self.df[
            self.df["Profit_Per_Hectare"] <= q_low
        ].copy()

        top_cohort["Cohort"] = "Top 10% (Profitable)"
        bottom_cohort["Cohort"] = "Bottom 10% (Loss-Making)"

        cohort_df = pd.concat([top_cohort, bottom_cohort], ignore_index=True)

        cohort_metrics = cohort_df.groupby("Cohort")[
            [
                "Seed_Quality_Score",
                "Yield_Tonnes_Ha",
                "Water_Efficiency_t_per_1000m3",
                "Soil_pH",
            ]
        ].mean()

        denominator = cohort_metrics.max() - cohort_metrics.min()
        norm_metrics = (cohort_metrics - cohort_metrics.min()).div(
            denominator.replace(0, np.nan)
        ).fillna(0)

        norm_metrics.T.plot(kind="bar", ax=ax, width=0.6)
        ax.set_title("Top 10% Profitable vs Bottom 10% Loss-Making Farms")
        ax.set_ylabel("Normalized Scale [0 - 1]")
        plt.xticks(rotation=0)

        fig.tight_layout()
        fig.savefig(self.figures_dir / "10_farm_profit_drivers_comparison.png")

        print(f"[SUCCESS] Figures exported to: {self.figures_dir}")

    def run_regression_and_drivers(self) -> Dict[str, Any]:
        """
        Fit an OLS model for Profit_Per_Hectare.

        Features are standardized so coefficients can be compared by magnitude.
        np.linalg.lstsq is used instead of directly inverting X'X, which is
        numerically more stable when multicollinearity exists.
        """
        print("\n" + "=" * 80)
        print("STEP 6: REGRESSION MODELING & STATISTICAL INFERENCE")
        print("=" * 80)

        feature_cols = [
            "Farm_Area_Hectares",
            "Rainfall_mm",
            "Avg_Temperature_C",
            "Humidity_pct",
            "Sunlight_Hours_Day",
            "Soil_pH",
            "Soil_Moisture_pct",
            "Nitrogen_kg_ha",
            "Phosphorus_kg_ha",
            "Potassium_kg_ha",
            "Fertilizer_kg_ha",
            "Pesticide_Litre_ha",
            "Seed_Quality_Score",
            "Yield_Tonnes_Ha",
        ]

        available_features = [c for c in feature_cols if c in self.df.columns]
        if not available_features:
            raise ValueError("No regression features are available in the dataset.")

        model_data = self.df[available_features + ["Profit_Per_Hectare"]].dropna()

        if len(model_data) <= len(available_features) + 1:
            raise ValueError(
                "Not enough observations to fit the regression model reliably."
            )

        X_raw = model_data[available_features].to_numpy(dtype=float)
        y = model_data["Profit_Per_Hectare"].to_numpy(dtype=float)

        # Standardize predictors.
        X_mean = X_raw.mean(axis=0)
        X_std = X_raw.std(axis=0, ddof=0)
        X_std[X_std == 0] = 1.0
        X_norm = (X_raw - X_mean) / X_std

        # Add intercept.
        X_design = np.column_stack([np.ones(len(X_norm)), X_norm])

        # Stable least-squares solution.
        beta, _, rank, _ = np.linalg.lstsq(X_design, y, rcond=None)

        if rank < X_design.shape[1]:
            print(
                "[WARNING] The design matrix is rank-deficient; "
                "some predictors may be strongly collinear."
            )

        predictions = X_design @ beta
        residuals = y - predictions

        ss_tot = np.sum((y - y.mean()) ** 2)
        ss_res = np.sum(residuals**2)

        r_squared = (
            1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan
        )

        rmse = float(np.sqrt(np.mean(residuals**2)))
        mae = float(np.mean(np.abs(residuals)))

        n = X_design.shape[0]
        p = X_design.shape[1]
        degrees_of_freedom = n - p

        if degrees_of_freedom > 0:
            sigma_sq = ss_res / degrees_of_freedom
            xtx_pinv = np.linalg.pinv(X_design.T @ X_design)
            var_beta = sigma_sq * xtx_pinv
            se_beta = np.sqrt(np.maximum(0, np.diag(var_beta)))
            t_stats = beta / np.where(se_beta > 0, se_beta, np.nan)
            p_values = 2 * stats.t.sf(
                np.abs(t_stats), df=degrees_of_freedom
            )
        else:
            se_beta = np.full(p, np.nan)
            t_stats = np.full(p, np.nan)
            p_values = np.full(p, np.nan)

        summary_data = [
            {
                "Feature": "Intercept",
                "Std_Coefficient": beta[0],
                "Std_Error": se_beta[0],
                "t_statistic": t_stats[0],
                "p_value": p_values[0],
            }
        ]

        for i, column in enumerate(available_features, start=1):
            summary_data.append(
                {
                    "Feature": column,
                    "Std_Coefficient": beta[i],
                    "Std_Error": se_beta[i],
                    "t_statistic": t_stats[i],
                    "p_value": p_values[i],
                }
            )

        model_df = pd.DataFrame(summary_data)
        model_df["Absolute_Std_Coefficient"] = model_df[
            "Std_Coefficient"
        ].abs()
        model_df = model_df.sort_values(
            "Absolute_Std_Coefficient",
            ascending=False,
        )

        reg_file = self.reports_dir / "ols_profit_driver_regression.csv"
        model_df.to_csv(reg_file, index=False)

        print(
            f"OLS Regression R-Squared : {r_squared:.4f} "
            f"({r_squared * 100:.2f}%)"
        )
        print(f"RMSE                     : INR {rmse:,.2f}")
        print(f"MAE                      : INR {mae:,.2f}")
        print("\nTop Predictor Coefficients:")
        print(
            model_df[
                ["Feature", "Std_Coefficient", "p_value"]
            ].head(6).to_string(index=False)
        )
        print(f"[SAVED] Regression report exported to: {reg_file}")

        return {
            "r_squared": r_squared,
            "rmse": rmse,
            "mae": mae,
            "coefficients": model_df,
        }

    def run_pipeline(self) -> None:
        """Execute all pipeline stages sequentially."""
        print("Starting End-to-End Agricultural Data Science Pipeline...")

        self.load_and_validate_data()
        self.compute_summary_statistics()
        self.generate_crop_analysis()
        self.generate_irrigation_analysis()
        self.generate_visualizations()
        self.run_regression_and_drivers()

        print("\n" + "=" * 80)
        print("PIPELINE COMPLETED SUCCESSFULLY")
        print(f"Output directory: {self.output_dir.resolve()}")
        print("=" * 80)


def main() -> None:
    """CLI entry point."""
    # Explicitly set the data path for Colab execution.
    # sys.argv parsing can be inconsistent in notebook environments.
    dataset_file = "/content/Cleaned_Agriculture_Data.json"
    pipeline = AgricultureDataPipeline(data_path=dataset_file)
    pipeline.run_pipeline()

    # Display the generated figures after the pipeline runs
    figures_path = pipeline.figures_dir
    if figures_path.exists() and figures_path.is_dir():
        image_files = sorted(list(figures_path.glob("*.png")))
        if image_files:
            print(f"\nDisplaying {len(image_files)} figures from {figures_path}:")
            for img_file in image_files:
                print(f"\n--- {img_file.name} ---")
                display(Image(filename=img_file))
        else:
            print(f"No image files (.png) found in {figures_path}")
    else:
        print(f"The directory {figures_path} does not exist or is not a directory. Please ensure the pipeline has run successfully.")


if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'matplotlib'